# Lezione 10 — VQE avanzato: Rotosolve e confronto di ottimizzatori

## Obiettivi di apprendimento

- Implementare Rotosolve e spiegare perché funziona sulle porte di rotazione
- Confrontare Rotosolve, Nelder-Mead e SPSA sullo stesso problema

Nella lezione precedente hai ottimizzato VQE con un metodo basato sul gradiente e con Nelder-Mead, un ottimizzatore a scatola nera generico. In questa lezione conosci Rotosolve, un ottimizzatore progettato specificamente per i circuiti quantistici parametrizzati con rotazioni, e lo confronti con Nelder-Mead e SPSA sullo stesso problema.

In [ ]:
!pip install -q cirq scipy noisyopt

## Perché è importante

Nella lezione precedente hai provato due ottimizzatori per VQE: un metodo basato sul gradiente (Adam) e un ottimizzatore a scatola nera generico (Nelder-Mead). Nessuno dei due sfrutta una proprietà matematica esclusiva delle porte di rotazione quantistiche. Rotosolve invece sì: a ogni iterazione trova il valore ottimo esatto di un parametro, senza avanzare con piccoli passi di gradiente né indovinare per tentativi ed errori. Capire Rotosolve ti dà l'intuizione su come progettare ottimizzatori su misura per problemi quantistici, invece di applicare sempre gli stessi strumenti classici.

## Perché un parametro di rotazione si comporta come un'onda

Fissa tutti i parametri del tuo ansatz tranne uno, e lascia che l'angolo di una porta `rx`, `ry` o `rz` vari liberamente. L'energia risultante `E(θ)` non è una curva qualsiasi: ha sempre la forma `E(θ) = A·cos(θ) + B·sin(θ) + C`, un'onda sinusoidale semplice. Questa proprietà è una conseguenza diretta del modo in cui le porte di rotazione trasformano lo stato quantistico. L'hai già verificato nella lezione precedente: la curva di energia che hai disegnato scandendo l'angolo da zero a due π aveva esattamente quella forma ondulata.

## Rotosolve: risolvere l'onda invece di approssimarla

Se sai che `E(θ)` è un'onda con quella forma esatta, non hai bisogno di piccoli passi di gradiente per trovarne il minimo. Ti basta valutare l'energia in tre punti precisi (un valore `φ` qualsiasi, e `φ + π/2` e `φ - π/2`) per ricavare i coefficienti A, B e C, e calcolare direttamente l'angolo che minimizza l'onda. Rotosolve ripete questo processo parametro per parametro, in passaggi successivi: a ogni passaggio porta ciascuno al suo valore ottimo esatto supponendo che gli altri restino fissi. Dopo diversi passaggi, l'insieme completo converge.

Il vantaggio rispetto a un metodo basato sul gradiente: ogni aggiornamento è un salto diretto all'ottimo (per quel parametro, con gli altri fissi), non un piccolo passo nella direzione di massima pendenza. Lo svantaggio: devi valutare il circuito più volte per ogni parametro e per ogni passaggio, quindi il costo per iterazione è maggiore.

## Confrontare tre ottimizzatori a parità di condizioni

Oltre a Rotosolve, questa lezione confronta due ottimizzatori classici a scatola nera:

- **Nelder-Mead**: esplora lo spazio dei parametri con un "simplesso" (una figura geometrica che si deforma) che si contrae verso la regione di energia più bassa; è robusto, ma può richiedere molte valutazioni quando l'ansatz ha molti parametri
- **SPSA** (approssimazione stocastica per perturbazione simultanea): a ogni passo perturba tutti i parametri insieme con una direzione casuale, valuta l'energia in quella direzione e stima il gradiente completo con solo due valutazioni del circuito, qualunque sia il numero totale di parametri. È utile quando valutare il circuito è costoso (per esempio, su hardware quantistico reale) e hai molti parametri da ottimizzare

Confrontare i tre ottimizzatori sullo stesso problema mostra che nessuno è universalmente migliore: la scelta dipende da quanti parametri hai, da quanto costa valutare il circuito e da quanto rumore c'è in quella valutazione.

## Primo esempio: Rotosolve ridotto

Di seguito trovi una versione ridotta di Rotosolve, con solo 30 iterazioni in modo che si esegua in pochi secondi. Riceve una funzione di energia qualsiasi e il numero di parametri, e restituisce l'angolo ottimo trovato per ciascuno.

In [ ]:
import numpy as np
import random

def rotosolve(funcion_energia, num_parametros, iteraciones=30):
    """Optimiza cada parámetro de forma analítica, ciclo a ciclo."""
    theta = np.random.uniform(-np.pi, np.pi, num_parametros)
    for _ in range(iteraciones):
        for d in range(num_parametros):
            phi = random.uniform(-np.pi, np.pi)
            theta_prueba = np.copy(theta)

            theta_prueba[d] = phi
            energia_phi = funcion_energia(theta_prueba)
            theta_prueba[d] = phi + np.pi / 2
            energia_mas = funcion_energia(theta_prueba)
            theta_prueba[d] = phi - np.pi / 2
            energia_menos = funcion_energia(theta_prueba)

            # Fórmula cerrada que resuelve el mínimo exacto de la onda A*cos + B*sin + C
            theta[d] = phi - np.pi / 2 - np.arctan2(
                2 * energia_phi - energia_mas - energia_menos, energia_mas - energia_menos
            )
    return {"fun": funcion_energia(theta), "params": theta}

## Secondo esempio: confronto completo su un Hamiltoniano a due qubit

Ora confronti Rotosolve, Nelder-Mead e SPSA su un Hamiltoniano a due qubit con due termini di Pauli, simulato direttamente con Cirq (senza passare per TensorFlow Quantum) per poter controllare la misura base per base. L'ansatz applica una rotazione `rx` e una `rz` per qubit e per strato, seguite da porte `CNOT` che mettono in entanglement i qubit vicini.

In [ ]:
import cirq
from scipy.optimize import minimize
from noisyopt import minimizeSPSA

def capa_ansatz(circuito, qubits, parametros):
    for qubit in qubits:
        circuito.append([cirq.rx(parametros[0]).on(qubit)])
        circuito.append([cirq.rz(parametros[1]).on(qubit)])
    for i in range(len(qubits) - 1):
        circuito.append([cirq.CNOT(qubits[i], qubits[i + 1])])
    return circuito

def rotar_a_base_de_medida(circuito, qubits, terminos_pauli):
    for i, qubit in enumerate(qubits):
        if terminos_pauli[i] == "x":
            circuito.append(cirq.ry(-np.pi / 2).on(qubit))
        elif terminos_pauli[i] == "y":
            circuito.append(cirq.rx(np.pi / 2).on(qubit))
    return circuito

def valor_esperado(circuito, qubits_medidos, repeticiones=1000):
    if len(qubits_medidos) == 0:
        return 1
    circuito.append(cirq.measure(*qubits_medidos, key="resultado"))
    simulador = cirq.Simulator()
    resultados = simulador.run(circuito, repetitions=repeticiones)
    medidas = resultados.measurements["resultado"]
    return np.mean([1 if sum(fila) % 2 == 0 else -1 for fila in medidas])

# Hamiltoniano de ejemplo: dos términos de Pauli sobre 2 qubits, con pesos fijos
terminos_hamiltoniano = [["x", "z"], ["z", "y"]]
pesos_hamiltoniano = [0.6, 0.3]
num_qubits, num_capas = 2, 2

def energia(parametros):
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    total = 0
    for termino, peso in zip(terminos_hamiltoniano, pesos_hamiltoniano):
        circuito = cirq.Circuit()
        for _ in range(num_capas):
            circuito = capa_ansatz(circuito, qubits, parametros)
        circuito = rotar_a_base_de_medida(circuito, qubits, termino)
        qubits_a_medir = [qubits[i] for i in range(len(termino)) if termino[i] != "i"]
        total += peso * valor_esperado(circuito, qubits_a_medir)
    return total

num_parametros = num_qubits * num_capas

resultado_nm = minimize(energia, np.random.uniform(0, 2 * np.pi, num_parametros), method="Nelder-Mead")
resultado_spsa = minimizeSPSA(energia, bounds=[[0, 2 * np.pi]] * num_parametros,
                               x0=np.random.uniform(0, 2 * np.pi, num_parametros), niter=100, paired=False)
resultado_roto = rotosolve(energia, num_parametros)

print("Nelder-Mead:", resultado_nm["fun"])
print("SPSA:", resultado_spsa["fun"])
print("Rotosolve:", resultado_roto["fun"])

Eseguendolo più volte, verifica che i tre ottimizzatori di solito convergono a energie simili, ma con un numero di valutazioni del circuito molto diverso: Rotosolve di solito richiede meno passaggi completi per stabilizzarsi, in cambio di una valutazione del circuito più volte per ogni parametro a ogni passaggio.

## Riepilogo dei punti chiave

- Il valore di attesa dell'energia, variando un singolo parametro di rotazione e lasciando fissi gli altri, ha sempre la forma di un'onda `A·cos(θ) + B·sin(θ) + C`
- Rotosolve sfrutta questa proprietà per calcolare l'angolo ottimo esatto di ogni parametro, valutando il circuito in soli tre punti per parametro
- SPSA stima il gradiente completo con solo due valutazioni del circuito, qualunque sia il numero di parametri, utile quando valutare il circuito è costoso
- Non esiste un ottimizzatore universalmente migliore: la scelta dipende dal numero di parametri, dal costo di valutazione del circuito e dal rumore presente in quella valutazione